# DVDT421. Using Different Computational Backends

<a href="https://colab.research.google.com/github/DeepTrackAI/DeepTrack2/blob/develop/tutorials/developers/DT421_backends.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install deeptrack  # Uncomment if running on Colab/Kaggle.

DeepTrack2 supports both NumPy and PyTorch for numerical computations. To help developers write backend-agnostic code, DeepTrack2 provides the `xp` proxy and a global `config` object.

This tutorial will show you how to use them to seamlessly switch between backends, select devices, and write robust, portable code.

This design can be easily extended to other libraries using the same interface, such as JAX and CuPy.

In [2]:
import deeptrack as dt

## What Is XP?

`xp` is a proxy that provides a unified interface to array operations between different
computational backends. We primarily use it to provide a unified interface to NumPy and PyTorch, but it also unifies operations between different array libraries, such as CuPy and JAX. 

The syntax is generally unified to be similar to NumPy, but there are some differences. Primarily, operations may accept some additional arguments (such as a device).

`xp` is intended for internal use. Users may use `xp` when writing custom features, but
are not forced to do so (it is strictly needed only if they want to be compatible with all backends).

For example, you can write some code using NumPy ...

In [3]:
import numpy as np

output_with_numpy = np.sum(np.random.randn(100, 100), keepdims=True)

print(f"type(output_with_numpy): {type(output_with_numpy)}")

type(output_with_numpy): <class 'numpy.ndarray'>


... and an equivalent version using `xp` (which uses the NumPy backend by default).

In [4]:
from deeptrack.backend import xp

output_with_xp = xp.sum(xp.random.randn(100, 100), keepdims=True)

print(f"type(output_with_xp): {type(output_with_xp)}")

type(output_with_xp): <class 'numpy.ndarray'>


## Changing the Backend

The `config.set_backend()` function allows you to switch between computational backends.
When you change the backend, all subsequent `xp` operations will use the specified backend.
This is useful for writing code that can run on different backends without changing the implementation.

In the example below, you can see how to use the same `xp` syntax to perform operations with both NumPy and PyTorch backends. Start by importing the `config` object ...

In [5]:
config = dt.backend.config  # Import the config object

... then set the backend to NumPy ...

In [6]:
config.set_backend("numpy")

output_with_numpy = xp.sum(xp.random.randn(100, 100), keepdims=True)

print(f"type(output_with_numpy): {type(output_with_numpy)}")

type(output_with_numpy): <class 'numpy.ndarray'>


... and finally change the backend to PyTorch.

In [7]:
config.set_backend("torch")

output_with_torch = xp.sum(xp.random.randn(100, 100), keepdims=True)

print(f"type(output_with_torch): {type(output_with_torch)}")

type(output_with_torch): <class 'torch.Tensor'>


**Note**: The native PyTorch syntax for the expression is different: (`keepdim` vs. `keepdims`, and `torch.randn` vs. `random.randn`). Moreover, for PyTorch `keepdim`, you also need to specify the dimensions you want to keep. This homogeneization is taken care in the `deeptrack.backend.array_api_compat_ext.torch.random` module. Importantly, this shows the value of `xp` when keeping interoperability between backends.

Using PyTorch, the code would read:

In [8]:
import torch

output = torch.sum(torch.randn(100, 100), dim=(0, 1), keepdim=True)

# output = torch.sum(torch.randn(100, 100), keepdim=True)  # This fails

print(f"type(output): {type(output)}")


type(output): <class 'torch.Tensor'>


## Using XP in Custom Features

It is advisable to use the `xp` module within your custom DeepTrack2 feature. By using `xp` instead of directly calling NumPy or PyTorch functions, your feature will automatically work with the currently selected backend, making your code more flexible and backend-agnostic.

### Creating an XP-Enabled Feature and Using It With Different Backends

The following example demonstrates how you can use the `xp` module within a custom DeepTrack2 feature. This is a simple `Zeros` feature that uses `xp.zeros()` to generate arrays of zeros.  This feature will automatically work with the current backend (NumPy or PyTorch) without any changes to the code.

In [9]:
class Zeros(dt.Feature):

    # Because this feature takes no inputs, it is not distributed.
    __distributed__ = False

    def __init__(self, shape: tuple[int, int]):
        super().__init__(shape=shape)

    def get(self, _, shape: tuple[int, int], **kwargs):
        return xp.zeros(shape)

Start by using this feature with a NumPy backend ...

In [10]:
config.set_backend("numpy")

feature_a = Zeros((100, 100))
output_a = feature_a()

print(type(output_a), "expected numpy.ndarray")

<class 'numpy.ndarray'> expected numpy.ndarray


... and then use it with a PyTorch backend (notice how you can switch backends globally using `config.set_backend()`).

In [11]:
config.set_backend("torch")

feature_b = Zeros((100, 100))
output_b = feature_b()

print(type(output_b), "expected torch.Tensor")

<class 'torch.Tensor'> expected torch.Tensor


### Features Keep Memory of their Backend

DeepTrack2 features maintain their backend configuration even when the global backend changes. This means that once a feature is created with a specific backend, it will continue to use that backend for all operations, regardless of subsequent changes to the global configuration.

The following code demonstrates this behavior by creating a feature with the NumPy backend and then changing the global backend to PyTorch. Notice how the feature still produces NumPy arrays despite the global backend change.

In [12]:
config.set_backend("numpy")

feature_a = Zeros((100, 100))
output_before_change = feature_a()
print(type(output_before_change), "expected numpy.ndarray")

config.set_backend("torch")
# Changed the global config, but the feature should still use the old backend
# since it was created before the global config was changed.

output_after_change = feature_a()
print(type(output_after_change), "expected numpy.ndarray")

<class 'numpy.ndarray'> expected numpy.ndarray
<class 'numpy.ndarray'> expected numpy.ndarray


### Changing the Backend of a Feature

DeepTrack2 features also provide methods to explicitly override their backend, regardless of the global configuration. 

You can use `.torch()` to force a feature to use PyTorch, or `.numpy()` to force it to use NumPy. This is particularly useful when you need specific backend functionality for certain operations while maintaining a different global backend.

The following code demonstrates how to override the backend for individual features. Notice how the output types match the explicitly specified backends, not the global configuration.

In [13]:
config.set_backend("numpy")

feature_a = Zeros((100, 100)).torch()
output_a = feature_a()
print(type(output_a), "expected torch.Tensor")

config.set_backend("torch")

feature_b = Zeros((100, 100)).numpy()
output_b = feature_b()
print(type(output_b), "expected numpy.ndarray")

<class 'torch.Tensor'> expected torch.Tensor
<class 'numpy.ndarray'> expected numpy.ndarray


## Chaining Features

When chaining features with different backends, you need to be careful about compatibility. DeepTrack2 features pass data from one feature to the next, and if the features use different backends, you may encounter errors.

**NOTE**: For computational efficiency, DeepTrack2 does not automatically convert between backends when chaining features.

The following code demonstrates how to create a feature pipeline using the `>>` operator to chain features together. Note how the backend of the first feature in the chain determines the backend of the entire pipeline's output.

In [14]:
class Sum(dt.Feature):

    def __init__(
        self,
        axis: int | None = None,
        keepdims: bool = False,
    ):
        super().__init__(axis=axis, keepdims=keepdims)

    def get(
        self,
        image: np.ndarray | torch.Tensor,
        axis: int | None = None,
        keepdims: bool = False,
        **kwargs,
    ):
        return xp.sum(image, axis=axis, keepdims=keepdims)

The following code demonstrates that the backend of chaiend features correctly uses
the intended backend.


In [15]:
config.set_backend("numpy")

feature1 = Zeros((100, 100)) >> Sum()
output1 = feature1()
print(type(output1), "expected numpy.ndarray")

config.set_backend("torch")

feature2 = Zeros((100, 100)) >> Sum()
output2 = feature2()
print(type(output2), "expected torch.Tensor")

<class 'numpy.float64'> expected numpy.ndarray
<class 'torch.Tensor'> expected torch.Tensor


Here, you can see that calling `feature.torch()` applies the PyTorch backend to the entire pipeline ...


In [16]:
config.set_backend("numpy")

feature = Zeros((100, 100)) >> Sum()

feature.torch()

output = feature()

print(type(output), "expected torch.Tensor")

<class 'torch.Tensor'> expected torch.Tensor


... while using `feature.numpy()` applies the NumPy backend to the entire pipeline.

In [17]:
config.set_backend("torch")

feature = Zeros((100, 100)) >> Sum()

feature.numpy()

output = feature()

print(type(output), "expected numpy.ndarray")

<class 'numpy.float64'> expected numpy.ndarray


The following code demonstrates how chaining features with different backends can lead to compatibility issues. When a feature with one backend tries to process data from a feature with a different backend, a `TypeError` occurs. This is because DeepTrack2 does not automatically convert between backends when chaining features.

In [18]:
zeros = Zeros((100, 100)).torch()

summer = Sum().numpy()

feature = zeros >> summer

try:
    output = feature()
except TypeError as e:  # TypeError as torch tensor is passed to summer
    print(e)

sum() received an invalid combination of arguments - got (out=NoneType, keepdims=bool, axis=NoneType, ), but expected one of:
 * (*, torch.dtype dtype)
 * (tuple of ints dim, bool keepdim, *, torch.dtype dtype)
 * (tuple of names dim, bool keepdim, *, torch.dtype dtype)



## Manually Converting Array Type

When working with multiple backends, you may need to manually convert between array types. The following code demonstrates a simple feature that converts any input to a NumPy array, which can be useful when you need to ensure compatibility between features using different backends. This approach provides explicit control over data type conversion in your processing pipeline.

In [19]:
class AsNumpy(dt.Feature):
    def get(self, image: np.ndarray | torch.Tensor, **kwargs):
        # Note that the use of np.asarray (not np.array)
        # to avoid unnecessary copies.
        return np.asarray(image)

Here, we use the `AsNumpy` feature to convert the torch Zeros output to a NumPy array, which can then be processed by the `Sum` feature.


In [20]:
zeros = Zeros((100, 100)).torch()
converter = AsNumpy()
summer = Sum().numpy()

feature = zeros >> converter >> summer

output = feature()

print(type(output), "expected numpy.ndarray")

<class 'numpy.float32'> expected numpy.ndarray


## Dispatching to the Correct Backend

Sometimes, you may need specific code paths for different backends.

The following example demonstrates how to dispatch to the correct backend using the `array_api_compat` library.

Start by implementing a feature that has two alternative codes for NumPy and PyTorch arrays ...

In [21]:
import array_api_compat as apc

class DispatchExample(dt.Feature):

    def foo(self, image: np.ndarray | torch.Tensor, **kwargs):
        if apc.is_numpy_array(image):
            return self.foo_numpy(image)
        elif apc.is_torch_array(image):
            return self.foo_torch(image)
        else:
            raise TypeError(
                f"Expected numpy.ndarray or torch.Tensor, got {type(image)}"
            )

    def foo_numpy(self, image: np.ndarray, **kwargs):
        print("Called NumPy version")
        return image

    def foo_torch(self, image: torch.Tensor, **kwargs):
        print("Called PyTorch version")
        return image

    def get(self, image: np.ndarray | torch.Tensor, **kwargs):
        return self.foo(image)


... and then use it with the PyTorch backend ...

In [22]:
feature = Zeros((100, 100)) >> DispatchExample()

feature.torch()

output = feature(zeros)

Called PyTorch version


... and with the NumPy backend.

In [23]:
feature = Zeros((100, 100)) >> DispatchExample()

feature.numpy()

out = feature(zeros)

Called NumPy version


## Compatability with `Image` and `properties`

`xp` and `Image` are compatible, and `properties` are preserved.

In [24]:
feature = Zeros((100, 100)) >> Sum()
feature.store_properties()
feature.torch()

x = feature()
x.properties, type(x._value)

/Users/giovannivolpe/Documents/GitHub/DeepLearningCrashCourse/py_env_book/lib/python3.10/site-packages/array_api_compat/torch/_aliases.py:344: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/python_arg_parser.cpp:298.)
  res = torch.sum(x, dtype=dtype, **kwargs)


([{'shape': (100, 100), 'name': 'Zeros'},
  {'axis': None, 'keepdims': False, 'name': 'Sum'},
  {'name': 'Chain'}],
 torch.Tensor)